<a href="https://colab.research.google.com/github/yourusername/yourrepo/blob/main/MNPS_Job_Classification_Improved_Two_Pass_v8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **MNPS Job Classification - Claude Opus Improved Two-Pass Approach v8.0**

> Enhanced two-pass classification with improved self-consistency and minimal post-processing  
> DSI DSSG + MNPS  

## **Version 8.0 Improvements**
- ✅ **Enhanced Self-Consistency**: Explicit alignment checking between justification and classification
- ✅ **Minimal Post-Processing**: Reduced rules that override LLM decisions
- ✅ **Better Role Distinctions**: Clear Coordinator vs Coach vs Manager differentiation
- ✅ **Proper Subgroup Handling**: Correct handling of roles without subgroups (Teacher, Principal)
- ✅ **Validation System**: Built-in checks for justification-classification alignment
- ✅ **Based on analysis showing Two-Pass v7.5.4 was most accurate**

## **1. Setup and Imports**

In [1]:
# ==== Core Imports ====
import os
import json
import shutil
import datetime as dt
import zipfile
from pathlib import Path
import pandas as pd
import numpy as np
import re
import time
import random
from google.colab import drive
from google.colab import userdata
from openai import OpenAI

# Install required packages if needed
!pip install -q openai tqdm

from tqdm import tqdm

print("✅ All imports successful")

✅ All imports successful


## **2. Mount Google Drive and Setup Directories**

In [2]:
# Mount Google Drive
drive.mount('/content/drive')

# Create unique run folder with v8 designation
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
run_folder = f"RUN_{timestamp}_v8"
base_path = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
run_path = base_path / run_folder
run_path.mkdir(parents=True, exist_ok=True)

# Create outputs subfolder
OUTPUTS_DIR = run_path / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Run folder: {run_path}")
print(f"📁 Outputs dir: {OUTPUTS_DIR}")

# Set working directory
RUN_ROOT = Path('/content')
print(f"📁 Working directory: {RUN_ROOT}")

Mounted at /content/drive
📁 Run folder: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251106_191954_v8
📁 Outputs dir: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251106_191954_v8/outputs
📁 Working directory: /content


## **3. Load Required Files**

Upload the following files to Colab:
- `Sample JDs.csv` - Job descriptions to classify
- `MNPS Prompt Resources.zip` - Contains MNPS roles, KSACs, and competencies
- `Ground Truth Masterfile.csv` (optional) - For validation

In [3]:
# Unzip MNPS Prompt Resources if needed
ZIP_FILE = RUN_ROOT / "MNPS Prompt Resources.zip"
if ZIP_FILE.exists():
    print(f"📦 Found {ZIP_FILE}, extracting...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_ref.extractall(RUN_ROOT)
    print("✅ Extracted MNPS Prompt Resources")
else:
    print("⚠️  MNPS Prompt Resources.zip not found - please upload it")

# Define file paths
BATCH_INPUT_CSV = RUN_ROOT / "Sample JDs.csv"
GT_MASTERFILE_CSV = RUN_ROOT / "Ground Truth Masterfile.csv"
MNPS_ROLES_CSV = RUN_ROOT / "MNPS Roles.csv"
MNPS_KSACS_CSV = RUN_ROOT / "MNPS KSACs.csv"
COMPETENCY_EXTENDED_CSV = RUN_ROOT / "Competency Extended Descriptions.csv"
KORN_FERRY_CSV = RUN_ROOT / "Korn_Ferry Lominger 38 Competencies.csv"

# Check for main input file
if not BATCH_INPUT_CSV.exists():
    print("⚠️  Sample JDs.csv not found - checking inside zip...")
    if ZIP_FILE.exists():
        with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
            if "Sample JDs.csv" in zip_ref.namelist():
                zip_ref.extract("Sample JDs.csv", RUN_ROOT)
                print("✅ Extracted Sample JDs.csv from zip")

print(f"\n📄 Files status:")
print(f"  Batch input: {'✅' if BATCH_INPUT_CSV.exists() else '❌'} {BATCH_INPUT_CSV}")
print(f"  MNPS Roles: {'✅' if MNPS_ROLES_CSV.exists() else '❌'} {MNPS_ROLES_CSV}")
print(f"  MNPS KSACs: {'✅' if MNPS_KSACS_CSV.exists() else '❌'} {MNPS_KSACS_CSV}")
print(f"  Competencies: {'✅' if COMPETENCY_EXTENDED_CSV.exists() else '❌'} {COMPETENCY_EXTENDED_CSV}")
print(f"  Korn Ferry: {'✅' if KORN_FERRY_CSV.exists() else '❌'} {KORN_FERRY_CSV}")

📦 Found /content/MNPS Prompt Resources.zip, extracting...
✅ Extracted MNPS Prompt Resources

📄 Files status:
  Batch input: ✅ /content/Sample JDs.csv
  MNPS Roles: ✅ /content/MNPS Roles.csv
  MNPS KSACs: ✅ /content/MNPS KSACs.csv
  Competencies: ✅ /content/Competency Extended Descriptions.csv
  Korn Ferry: ✅ /content/Korn_Ferry Lominger 38 Competencies.csv


## **4. Load and Explore Data**

In [7]:
# Load all dataframes
df = pd.read_csv(BATCH_INPUT_CSV, encoding='latin1')
roles_df = pd.read_csv(MNPS_ROLES_CSV)
ksacs_df = pd.read_csv(MNPS_KSACS_CSV)
competency_df = pd.read_csv(COMPETENCY_EXTENDED_CSV, encoding='latin1')
korn_ferry_df = pd.read_csv(KORN_FERRY_CSV, encoding='latin1')

print(f"✅ Loaded {len(df)} job descriptions")
print(f"✅ Loaded {len(roles_df)} MNPS roles")
print(f"✅ Loaded {len(ksacs_df)} KSACs")
print(f"✅ Loaded {len(competency_df)} competency descriptions")
print(f"✅ Loaded {len(korn_ferry_df)} Korn Ferry competencies")

# Display sample of job descriptions
print("\n📊 Sample job descriptions:")
print(df[['Job Code', 'Job Description Name']].head())

✅ Loaded 43 job descriptions
✅ Loaded 62 MNPS roles
✅ Loaded 310 KSACs
✅ Loaded 38 competency descriptions
✅ Loaded 38 Korn Ferry competencies

📊 Sample job descriptions:
   Job Code                     Job Description Name
0     80259                   Mgr Operations Routing
1     89101                    Asst Therapy Physical
2     81473                                Coord ACT
3     83350                   Dir Student Assignment
4     85111  Translator School Based Parent Outreach


## **5. Define Valid Roles and Normalization Functions**

In [9]:
# Extract valid roles from MNPS Roles file
VALID_ROLES = sorted(roles_df['Roles'].dropna().unique().tolist())
print(f"✅ Found {len(VALID_ROLES)} valid MNPS roles:")
print(f"   {', '.join(VALID_ROLES[:10])}...")

# Define role categories (v8 improvement: clearer categorization)
EXECUTIVE_ROLES = ['Coordinator', 'Principal', 'Director', 'Manager']
NO_SUBGROUP_ROLES = ['Teacher', 'Assistant Principal', 'Principal']
RARELY_LEAD_ROLES = ['Director', 'Principal', 'Manager', 'Coordinator']

print(f"\n📋 Role Categories:")
print(f"   Executive roles: {', '.join(EXECUTIVE_ROLES)}")
print(f"   Roles typically without subgroups: {', '.join(NO_SUBGROUP_ROLES)}")

def normalize_minor(minor_str):
    """Normalize minor role to standard format."""
    if not minor_str or minor_str == 'nan' or pd.isna(minor_str):
        return ''

    minor_str = str(minor_str).strip()

    # Handle all variations
    conversions = {
        '1': 'I', 'i': 'I', 'I': 'I',
        '2': 'II', 'ii': 'II', 'II': 'II',
        '3': 'III', 'iii': 'III', 'III': 'III',
        'Lead': 'Lead', 'lead': 'Lead', 'LEAD': 'Lead',
        '': ''
    }

    return conversions.get(minor_str, minor_str)

print("\n✅ Normalization functions defined")

✅ Found 62 valid MNPS roles:
   Accountant, Administrative Assistant, Advisor, Agent, Aide, Analyst, Architect (Facility-Focused), Architect (Technology-Focused), Assistant, Assistant Principal...

📋 Role Categories:
   Executive roles: Coordinator, Principal, Director, Manager
   Roles typically without subgroups: Teacher, Assistant Principal, Principal

✅ Normalization functions defined


## **6. Build Comprehensive KSACs Text**

In [10]:
def build_ksacs_text():
    """Build comprehensive KSACs text from all MNPS resources."""
    ksacs_text = "MNPS Knowledge, Skills, Abilities, and Competencies (KSACs):\n"

    # Clean column names
    ksacs_df.columns = ksacs_df.columns.str.strip()
    competency_df.columns = competency_df.columns.str.strip()
    korn_ferry_df.columns = korn_ferry_df.columns.str.strip()

    # Add role-specific KSACs
    role_col = next((col for col in ksacs_df.columns if 'role' in col.lower()), None)
    ksacs_col = next((col for col in ksacs_df.columns if 'ksacs' in col.lower()), None)

    if role_col and ksacs_col:
        for _, row in ksacs_df.iterrows():
            role = row.get(role_col, '')
            ksacs = row.get(ksacs_col, '')
            if role and ksacs:
                ksacs_text += f"**{role}**:\n{ksacs}\n"

    # Add competency descriptions
    comp_col = next((col for col in competency_df.columns if 'competency' in col.lower()), None)
    desc_col = next((col for col in competency_df.columns if 'description' in col.lower()), None)

    if comp_col and desc_col:
        ksacs_text += "\n**Competency Extended Descriptions**:\n"
        for _, row in competency_df.iterrows():
            competency = row.get(comp_col, '')
            description = row.get(desc_col, '')
            if competency and description:
                ksacs_text += f"- {competency}: {description}\n"

    # Add Korn Ferry competencies
    kf_comp_col = next((col for col in korn_ferry_df.columns if 'competency' in col.lower()), None)
    kf_def_col = next((col for col in korn_ferry_df.columns
                        if 'description' in col.lower() or 'definition' in col.lower()), None)

    if kf_comp_col and kf_def_col:
        ksacs_text += "\n**Korn Ferry Lominger 38 Competencies**:\n"
        for _, row in korn_ferry_df.iterrows():
            competency = row.get(kf_comp_col, '')
            definition = row.get(kf_def_col, '')
            if competency and definition:
                ksacs_text += f"- {competency}: {definition}\n"

    return ksacs_text

KSACS_TEXT = build_ksacs_text()
print(f"✅ Built comprehensive KSACs text ({len(KSACS_TEXT)} characters)")
print("✅ Includes all 4 critical MNPS resource documents")

✅ Built comprehensive KSACs text (37793 characters)
✅ Includes all 4 critical MNPS resource documents


## **7. Define Enhanced Prompts (v8 Improvements)**

In [11]:
# ==== Enhanced Zero Shot Prompt ====
ZERO_SHOT_PROMPT = """Objective: Classify this job based on its functions and requirements, NOT its title.

Process:
- Analyze job attributes: Education, Work Experience, Essential Functions, KSAs
- Match to MNPS role classifications based on actual work performed
- Provide clear justification based on job attributes and MNPS standards

CRITICAL ROLE DISTINCTIONS:
- **Technician**: Hands-on technical work, equipment maintenance, repair, installation
- **Specialist**: Specialized domain knowledge (avoid overuse - prefer specific roles when possible)
- **Analyst**: Data analysis, research, evaluation, assessment, reporting
- **Coordinator**: Program coordination, organization, facilitation, liaison work
- **Coach**: Instructional support, mentoring, professional development, co-teaching
- **Manager**: Strategic planning, policy development, budget oversight, supervision
- **Supervisor**: Primarily people management, no post-high school education required
- **Assistant**: Supporting role in specialized function under supervision
- **Teacher**: Direct instruction of K-12 students
- **Principal/Assistant Principal**: School leadership roles

MINOR SUB-GROUP GUIDELINES:
- **I**: Entry-level, basic complexity, minimal experience required
- **II**: Intermediate complexity and responsibility
- **III**: Advanced KSACs, senior-level expertise
- **Lead**: Leads teams/projects (rare for executive roles)
- **No subgroup (blank)**: Some roles like Teacher, Principal, Assistant Principal typically have no subgroup

Output Requirements:
Return JSON with:
- new_job_title: Descriptive title incorporating role and level
- major_role_group: From approved MNPS roles
- minor_sub_group: I, II, III, Lead, or blank (empty string)
- grouping_justification: Detailed explanation that MUST mention the chosen major role"""

# ==== Enhanced Self-Consistency Prompt (v8 key improvement) ====
SELF_CONSISTENCY_PROMPT = """TASK: Review your classification and ensure PERFECT alignment between justification and classification.

CRITICAL CHECKS:

1. **Justification-Classification Alignment**:
   - If justification describes "coordination/organizing/facilitating" → major_role_group MUST be "Coordinator"
   - If justification describes "instructional support/mentoring/co-teaching" → major_role_group MUST be "Coach"
   - If justification describes "strategic planning/budget/supervision" → major_role_group MUST be "Manager"
   - If justification describes "data analysis/research/evaluation" → major_role_group MUST be "Analyst"
   - If justification describes "hands-on technical/repair/maintenance" → major_role_group MUST be "Technician"
   - If justification describes "therapy/treatment/intervention" → major_role_group MUST be "Therapist"
   - If justification describes "assisting/supporting under supervision" → major_role_group MUST be "Assistant"

2. **Minor Subgroup Rules**:
   - Teacher: Usually NO subgroup (empty string) unless explicitly "Lead Teacher"
   - Assistant Principal: Usually NO subgroup (empty string)
   - Principal: Usually NO subgroup (empty string)
   - Executive roles (Director, Manager, Coordinator): Rarely "Lead", usually I/II/III
   - Entry positions with <2 years experience: Usually "I"
   - Intermediate with 3-5 years: Usually "II"
   - Senior/advanced with 5+ years: Usually "III"

3. **Common Errors to Fix**:
   - Coordinator vs Coach confusion (Coordinators organize programs; Coaches provide instruction)
   - Manager vs Coordinator confusion (Managers have budget/strategic duties; Coordinators facilitate)
   - Therapist vs Assistant confusion (Therapists provide treatment; Assistants support under supervision)
   - Overuse of "Specialist" (prefer specific roles like Analyst, Technician)

INSTRUCTIONS:
- If ANY mismatch exists between justification and classification, FIX the classification
- Keep the justification text unchanged
- Return corrected JSON with same structure
- Ensure major_role_group appears in the justification"""

print("✅ Enhanced prompts defined with v8 improvements")
print("   - Stronger alignment checking")
print("   - Clearer role distinctions")
print("   - Better subgroup guidance")

✅ Enhanced prompts defined with v8 improvements
   - Stronger alignment checking
   - Clearer role distinctions
   - Better subgroup guidance


## **8. OpenAI API Setup**

In [12]:
# Get API key from Colab's secrets
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Initialize OpenAI client
client = OpenAI()

# Use GPT-4o for consistency with previous versions
MODEL_ID = "gpt-4o-2024-11-20"

print(f"✅ OpenAI client initialized")
print(f"✅ Using model: {MODEL_ID}")

def call_llm_json_with_retry(prompt: str, model: str = None, max_retries: int = 3) -> dict:
    """Call OpenAI API with JSON response and exponential backoff for rate limiting."""
    if model is None:
        model = MODEL_ID

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
                temperature=0.2
            )
            return json.loads(response.choices[0].message.content)

        except Exception as e:
            error_str = str(e).lower()

            if "429" in error_str or "rate limit" in error_str or "quota" in error_str:
                if attempt < max_retries - 1:
                    wait_time = (2 ** attempt) + random.uniform(0, 1)
                    print(f"⚠️  Rate limit hit, waiting {wait_time:.1f}s before retry {attempt + 1}/{max_retries}")
                    time.sleep(wait_time)
                    continue
                else:
                    print(f"❌ Max retries reached for rate limiting")
                    raise e
            else:
                print(f"❌ Non-rate limiting error: {e}")
                raise e

    raise Exception("Max retries reached")

print("✅ API call function with retry logic defined")

✅ OpenAI client initialized
✅ Using model: gpt-4o-2024-11-20
✅ API call function with retry logic defined


## **9. Minimal Post-Processing (v8 Key Improvement)**

In [13]:
def minimal_post_processing(major_role, minor_role, job_text, justification):
    """Apply only essential post-processing rules - v8 improvement."""

    # 1. Normalize minor role format
    minor_role = normalize_minor(minor_role)

    # 2. Handle roles that typically don't have sub-groups
    if major_role in NO_SUBGROUP_ROLES:
        # Only keep 'Lead' designation if explicitly mentioned
        if minor_role and minor_role != 'Lead':
            if 'lead' not in job_text.lower() and 'lead' not in justification.lower():
                minor_role = ''

    # 3. Fix ONLY clear mismatches between justification and role
    justification_lower = justification.lower()

    # Only override if there's a VERY clear mismatch
    if major_role == 'Coach' and 'coach' not in justification_lower and 'instruct' not in justification_lower:
        if 'coordinat' in justification_lower:
            major_role = 'Coordinator'
    elif major_role == 'Coordinator' and 'coordinat' not in justification_lower:
        if 'coach' in justification_lower or 'instruct' in justification_lower:
            major_role = 'Coach'

    # 4. Executive roles rarely have "Lead"
    if major_role in EXECUTIVE_ROLES and minor_role == 'Lead':
        if major_role in ['Director', 'Principal']:
            minor_role = 'III'
        else:
            minor_role = 'II'

    return major_role, minor_role

def validate_classification(major_role, minor_role, justification):
    """Validate that classification matches justification."""
    issues = []
    justification_lower = justification.lower()

    # Role keywords to check
    role_keywords = {
        'Coordinator': ['coordinat', 'organiz', 'facilitat', 'liaison'],
        'Coach': ['coach', 'instruct', 'mentor', 'professional development'],
        'Manager': ['manag', 'strategic', 'supervis', 'budget', 'policy'],
        'Analyst': ['analy', 'data', 'research', 'evaluat', 'assess'],
        'Technician': ['technical', 'hands-on', 'repair', 'maintain', 'equipment'],
        'Teacher': ['teach', 'instruct', 'lesson', 'classroom', 'student'],
        'Assistant': ['assist', 'support', 'help', 'under supervision'],
        'Specialist': ['specializ', 'expert'],
        'Therapist': ['therap', 'treatment', 'intervention'],
        'Principal': ['principal', 'school leader'],
        'Director': ['director', 'oversee', 'department']
    }

    # Check if role appears in justification
    if major_role in role_keywords:
        found_keyword = False
        for keyword in role_keywords[major_role]:
            if keyword in justification_lower:
                found_keyword = True
                break

        if not found_keyword and major_role.lower() not in justification_lower:
            issues.append(f"Role '{major_role}' not aligned with justification")

    return issues

print("✅ Minimal post-processing functions defined")
print("   - Reduced rule overrides")
print("   - Trust LLM self-consistency more")
print("   - Validation without aggressive correction")

✅ Minimal post-processing functions defined
   - Reduced rule overrides
   - Trust LLM self-consistency more
   - Validation without aggressive correction


## **10. Main Two-Pass Processing Function (v8 Enhanced)**

In [14]:
def process_job_description_v8(row_idx: int, row: pd.Series) -> dict:
    """Improved two-pass processing with better self-consistency - v8."""

    # Get original job title for reference (but don't use in classification)
    job_title_original = row.get('Job Description Name', '')

    # Build job text (ignoring title)
    job_text = f"""Position Summary: {row.get('Position Summary', '')}
Essential Functions: {row.get('Essential Functions', '')}
Work Experience: {row.get('Work Experience', '')}
Education: {row.get('Education', '')}
Licenses and Certifications: {row.get('Licenses and Certifications', '')}
Knowledge, Skills and Abilities: {row.get('Knowledge, Skills and Abilities', '')}"""

    # === PASS 1: Initial Classification ===
    pass1_prompt = f"""{ZERO_SHOT_PROMPT}

Available MNPS Roles: {', '.join(VALID_ROLES)}

{KSACS_TEXT}

Job Description to Classify:
{job_text}

IMPORTANT:
- Ignore the job title completely
- Base classification solely on job attributes
- Your justification MUST mention the major role you select

Return JSON with: new_job_title, major_role_group, minor_sub_group, grouping_justification"""

    try:
        # Get initial classification
        pass1_response = call_llm_json_with_retry(pass1_prompt)

        raw_major = pass1_response.get('major_role_group', 'Other')
        raw_minor = pass1_response.get('minor_sub_group', '')
        justification = pass1_response.get('grouping_justification', 'No justification provided')

        # === PASS 2: Self-Consistency Check ===
        pass2_prompt = f"""{SELF_CONSISTENCY_PROMPT}

Your Previous Output:
{{
  "major_role_group": "{raw_major}",
  "minor_sub_group": "{raw_minor}",
  "grouping_justification": {json.dumps(justification)}
}}

Review for mismatches and return the corrected JSON.
Remember: The justification must describe why you chose the major_role_group."""

        pass2_response = call_llm_json_with_retry(pass2_prompt)

        # Extract corrected values
        final_major = pass2_response.get('major_role_group', raw_major)
        final_minor = pass2_response.get('minor_sub_group', raw_minor)
        final_justification = pass2_response.get('grouping_justification', justification)

        # Apply MINIMAL post-processing (v8 key improvement)
        final_major, final_minor = minimal_post_processing(
            final_major, final_minor, job_text, final_justification
        )

        # Validate alignment
        validation_issues = validate_classification(final_major, final_minor, final_justification)

        # Construct title
        if final_minor:
            new_job_title = f"{final_major} {final_minor}"
        else:
            new_job_title = final_major

        return {
            'source_row_index': row_idx,
            'job_title_original': job_title_original,
            'new_job_title': new_job_title,
            'major_role_group': final_major,
            'minor_sub_group': final_minor,
            'grouping_justification': final_justification,
            'validation_issues': '; '.join(validation_issues) if validation_issues else 'None',
            'pass1_classification': f"{raw_major} {raw_minor}".strip(),
            'classification_changed': f"{raw_major} {raw_minor}".strip() != f"{final_major} {final_minor}".strip(),
            'model_used': MODEL_ID
        }

    except Exception as e:
        print(f"❌ Error processing row {row_idx}: {e}")
        return {
            'source_row_index': row_idx,
            'job_title_original': job_title_original,
            'new_job_title': 'Error',
            'major_role_group': 'Other',
            'minor_sub_group': 'I',
            'grouping_justification': f'Error: {str(e)}',
            'validation_issues': 'Processing error',
            'pass1_classification': 'Error',
            'classification_changed': False,
            'model_used': MODEL_ID
        }

print("✅ Main processing function defined with v8 enhancements")

✅ Main processing function defined with v8 enhancements


## **11. Batch Processing**

In [16]:
# Process all job descriptions
print("\n🚀 Starting improved two-pass batch processing (v8)...")
print(f"   Processing {len(df)} job descriptions")
print(f"   Using model: {MODEL_ID}")
print(f"   Rate limiting: 0.3s between calls\n")

results = []
validation_issue_count = 0

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing jobs"):
    result = process_job_description_v8(idx, row)
    results.append(result)

    # Track validation issues
    if result['validation_issues'] != 'None':
        validation_issue_count += 1
        print(f"\n⚠️  Row {idx}: {result['validation_issues']}")

    # Rate limiting protection
    time.sleep(0.3)

print(f"\n✅ Processing complete!")
print(f"   Jobs processed: {len(results)}")
print(f"   Validation issues found: {validation_issue_count}")


🚀 Starting improved two-pass batch processing (v8)...
   Processing 43 job descriptions
   Using model: gpt-4o-2024-11-20
   Rate limiting: 0.3s between calls



Processing jobs: 100%|██████████| 43/43 [19:18<00:00, 26.94s/it]


✅ Processing complete!
   Jobs processed: 43
   Validation issues found: 0


## **12. Save Results and Generate Reports**

In [17]:
# Convert results to DataFrame
results_df = pd.DataFrame(results)

# Save main results
output_path = OUTPUTS_DIR / "Job_Classifications_v8_Improved.csv"
results_df.to_csv(output_path, index=False)
print(f"✅ Saved results to: {output_path}")

# Generate summary statistics
print("\n📊 Classification Summary:")
print("=" * 50)
print(f"Total jobs processed: {len(results_df)}")
print(f"Unique major roles: {results_df['major_role_group'].nunique()}")
print(f"Unique minor roles: {results_df['minor_sub_group'].nunique()}")

# Self-consistency analysis
changed_classifications = results_df[results_df['classification_changed'] == True]
print(f"\nSelf-consistency corrections: {len(changed_classifications)} ({len(changed_classifications)/len(results_df)*100:.1f}%)")

# Validation issues
issues_df = results_df[results_df['validation_issues'] != 'None']
print(f"Classifications with validation issues: {len(issues_df)} ({len(issues_df)/len(results_df)*100:.1f}%)")

# Major role distribution
print("\nTop 10 Major Roles:")
print(results_df['major_role_group'].value_counts().head(10))

# Minor role distribution
print("\nMinor Role Distribution:")
minor_counts = results_df['minor_sub_group'].value_counts()
minor_counts['(blank)'] = (results_df['minor_sub_group'] == '').sum()
print(minor_counts.sort_values(ascending=False))

✅ Saved results to: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251106_191954_v8/outputs/Job_Classifications_v8_Improved.csv

📊 Classification Summary:
Total jobs processed: 43
Unique major roles: 11
Unique minor roles: 5

Self-consistency corrections: 9 (20.9%)
Classifications with validation issues: 0 (0.0%)

Top 10 Major Roles:
major_role_group
Coordinator    11
Manager         7
Analyst         5
Coach           5
Assistant       3
Technician      3
Teacher         3
Therapist       3
Translator      1
Driver          1
Name: count, dtype: int64

Minor Role Distribution:
minor_sub_group
II         19
           15
(blank)    15
I           5
III         3
Lead        1
Name: count, dtype: int64


## **13. Problem Area Analysis**

In [18]:
print("\n🔍 Problem Area Analysis:")
print("=" * 50)

# Coordinator/Coach analysis
coord_coach = results_df[results_df['major_role_group'].isin(['Coordinator', 'Coach'])]
coord_coach_issues = coord_coach[coord_coach['validation_issues'] != 'None']
print(f"\nCoordinator/Coach classifications: {len(coord_coach)}")
print(f"  - With validation issues: {len(coord_coach_issues)}")
if len(coord_coach) > 0:
    print("  - Distribution:")
    print(f"    Coordinator: {(coord_coach['major_role_group'] == 'Coordinator').sum()}")
    print(f"    Coach: {(coord_coach['major_role_group'] == 'Coach').sum()}")

# Executive roles with "Lead"
exec_df = results_df[results_df['major_role_group'].isin(EXECUTIVE_ROLES)]
exec_lead = exec_df[exec_df['minor_sub_group'] == 'Lead']
print(f"\nExecutive roles: {len(exec_df)}")
print(f"  - With 'Lead' designation: {len(exec_lead)}")
if len(exec_lead) > 0:
    print("  - Examples:")
    for _, row in exec_lead.head(3).iterrows():
        print(f"    {row['job_title_original']}: {row['major_role_group']} Lead")

# Roles without subgroups
for role in NO_SUBGROUP_ROLES:
    role_df = results_df[results_df['major_role_group'] == role]
    if len(role_df) > 0:
        no_subgroup = role_df[role_df['minor_sub_group'] == '']
        with_subgroup = role_df[role_df['minor_sub_group'] != '']
        print(f"\n{role}: {len(role_df)} total")
        print(f"  - Without subgroup (correct): {len(no_subgroup)} ({len(no_subgroup)/len(role_df)*100:.1f}%)")
        print(f"  - With subgroup: {len(with_subgroup)} ({len(with_subgroup)/len(role_df)*100:.1f}%)")
        if len(with_subgroup) > 0:
            print(f"    Subgroups: {with_subgroup['minor_sub_group'].value_counts().to_dict()}")


🔍 Problem Area Analysis:

Coordinator/Coach classifications: 16
  - With validation issues: 0
  - Distribution:
    Coordinator: 11
    Coach: 5

Executive roles: 18
  - With 'Lead' designation: 0

Teacher: 3 total
  - Without subgroup (correct): 3 (100.0%)
  - With subgroup: 0 (0.0%)


## **14. Export Detailed Reports**

In [19]:
# Create summary statistics DataFrame
summary_stats = {
    'metric': [
        'total_jobs',
        'unique_major_roles',
        'unique_minor_roles',
        'self_consistency_corrections',
        'validation_issues',
        'coordinator_coach_issues',
        'executive_with_lead',
        'teacher_without_subgroup',
        'principal_without_subgroup'
    ],
    'value': [
        len(results_df),
        results_df['major_role_group'].nunique(),
        results_df['minor_sub_group'].nunique(),
        len(changed_classifications),
        len(issues_df),
        len(coord_coach_issues) if 'coord_coach_issues' in locals() else 0,
        len(exec_lead) if 'exec_lead' in locals() else 0,
        len(results_df[(results_df['major_role_group'] == 'Teacher') & (results_df['minor_sub_group'] == '')]),
        len(results_df[(results_df['major_role_group'] == 'Principal') & (results_df['minor_sub_group'] == '')])
    ]
}

summary_df = pd.DataFrame(summary_stats)
summary_df.to_csv(OUTPUTS_DIR / "summary_stats_v8.csv", index=False)
print("✅ Summary statistics saved")

# Export validation issues for review
if len(issues_df) > 0:
    issues_export = issues_df[['source_row_index', 'job_title_original', 'major_role_group',
                                'minor_sub_group', 'validation_issues', 'grouping_justification']]
    issues_export.to_csv(OUTPUTS_DIR / "validation_issues_v8.csv", index=False)
    print("✅ Validation issues exported for review")

# Export self-consistency corrections for analysis
if len(changed_classifications) > 0:
    corrections_export = changed_classifications[['source_row_index', 'job_title_original',
                                                   'pass1_classification', 'new_job_title']]
    corrections_export.to_csv(OUTPUTS_DIR / "self_consistency_corrections_v8.csv", index=False)
    print("✅ Self-consistency corrections exported")

print(f"\n📁 All outputs saved to: {OUTPUTS_DIR}")
print("\n✨ Processing complete! Review the outputs for detailed results.")

✅ Summary statistics saved
✅ Self-consistency corrections exported

📁 All outputs saved to: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251106_191954_v8/outputs

✨ Processing complete! Review the outputs for detailed results.


## **15. Optional: Compare with Previous Versions**

If you have results from previous versions, you can load and compare them here.

In [20]:
# Optional: Load and compare with previous results
# Uncomment and modify paths as needed

# # Load previous results
# prev_results_path = "/path/to/previous/results.csv"
# if Path(prev_results_path).exists():
#     prev_results = pd.read_csv(prev_results_path)
#
#     # Merge on source_row_index
#     comparison = results_df.merge(
#         prev_results,
#         on='source_row_index',
#         suffixes=('_v8', '_prev')
#     )
#
#     # Calculate agreement
#     major_match = comparison['major_role_group_v8'] == comparison['major_role_group_prev']
#     minor_match = comparison['minor_sub_group_v8'] == comparison['minor_sub_group_prev']
#
#     print("\n📊 Comparison with Previous Version:")
#     print(f"Major role agreement: {major_match.mean()*100:.1f}%")
#     print(f"Minor role agreement: {minor_match.mean()*100:.1f}%")
#     print(f"Complete agreement: {(major_match & minor_match).mean()*100:.1f}%")
#
#     # Show disagreements
#     disagreements = comparison[~(major_match & minor_match)]
#     print(f"\nDisagreements: {len(disagreements)}")
#     if len(disagreements) > 0:
#         disagreements[['job_title_original_v8', 'major_role_group_v8', 'major_role_group_prev']].head()

## **Summary and Next Steps**

### ✅ **What This Notebook Does:**
1. Implements enhanced two-pass classification with improved self-consistency
2. Uses minimal post-processing to avoid overriding good LLM decisions
3. Validates alignment between justifications and classifications
4. Properly handles roles without subgroups (Teacher, Principal)
5. Provides detailed analysis and reporting

### 📈 **Expected Improvements:**
- Better Coordinator vs Coach distinction
- Correct handling of Teacher/Principal subgroups
- Higher justification-classification alignment
- Fewer executive roles with "Lead" designation

### 🎯 **Next Steps:**
1. Review the validation issues to identify remaining problems
2. Compare results with ground truth if available
3. Fine-tune prompts based on specific error patterns
4. Consider ensemble approaches for difficult cases

### 📝 **Notes:**
- Rate limiting is set to 0.3s between API calls
- Temperature is set to 0.2 for consistency
- All outputs are saved to Google Drive for persistence